In [90]:
R = np.array(
    [
        [0.06, 0.02],
        [-0.02, 0.04],
        [0.04, -0.01]
    ]
)

S, N = R.shape

p = np.ones(S)/S

In [92]:
mu_A = np.sum(R[:, 0])/S
mu_B = np.sum(R[:, 1])/S

spread = mu_A - mu_B

In [112]:
A_eq = np.array([R[0, 0] - R[0, 1], R[1, 0] - R[1, 1], R[2, 0] - R[2, 1]])
b_eq = np.array([0.03])

In [190]:
def solve_entropy_pooling(R, p=None, A_eq=None, b_eq=None, A_ineq=None, b_ineq=None):

    S, N = R.shape
    if p is None:
        p = np.ones(S) / S

    A_list, b_list, constraints = [], [], []

    k_eq = 0
    if A_eq is not None and b_eq is not None:
        A_eq_clean = np.atleast_2d(A_eq)
        b_eq_clean = np.atleast_1d(b_eq)
        A_list.append(A_eq_clean)
        b_list.append(b_eq_clean)
        k_eq = A_eq_clean.shape[0]

    k_ineq = 0
    if A_ineq is not None and b_ineq is not None:
        A_ineq_clean = np.atleast_2d(A_ineq)
        b_ineq_clean = np.atleast_1d(b_ineq)
        A_list.append(A_ineq_clean)
        b_list.append(b_ineq_clean)
        k_ineq = A_ineq_clean.shape[0]

    A = np.vstack(A_list)
    b = np.concatenate(b_list)
    K = k_eq + k_ineq

    lambda_var = cp.Variable(K)

    if k_ineq > 0:
        constraints.append(lambda_var[k_eq:] >= 0)

    exponent_terms = -lambda_var @ A + np.log(p)
    obj_fn = cp.log_sum_exp(exponent_terms) + lambda_var @ b

    prob = cp.Problem(cp.Minimize(obj_fn), constraints)
    prob.solve(solver=cp.ECOS)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Optimization failed. Status: {prob.status}")

    opt_lambda = lambda_var.value

    q = p * np.exp(-opt_lambda @ A)
    q /= np.sum(q)

    mu_post = q @ R
    R_dev = R - mu_post
    cov_post = (R_dev.T * q) @ R_dev

    return q, mu_post, cov_post

In [ ]:
solve_entropy_pooling(R, p=p, A_eq=A_eq, b_eq=b_eq)